In [1]:
# GOBERNANZA Y ÉTICA DE DATOS - TRÁFICO DE RED
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

print("Librerias importadas ------")

from Configuracion import CARGADOS, PROCESADOS, CSV_GENERADOS, IMPLEMENTACION_GOBERNANZA, ANALITICOS_PARQUET, ANALITICOS_RESULTADOS
from Configuracion import RECURSOS

# CREAR CARPETAS PARA GOBERNANZA
DATA_GOV = IMPLEMENTACION_GOBERNANZA
DOCS_DIR = CSV_GENERADOS

DATA_GOV.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datos gobernanza: {DATA_GOV}")
print(f"Documentos: {DOCS_DIR}")

Librerias importadas ------
Datos gobernanza: ..\Datos\Implementacion_Gobernanza
Documentos: ..\Datos\CSV_Generados


In [2]:
# CARGAR DATASET REAL (PARQUET)
ruta_parquet = ANALITICOS_PARQUET / "dataset_enriquecido_final.parquet"

try:
    df_red = pd.read_parquet(ruta_parquet)
    print(f"Dataset de red cargado. Forma: {df_red.shape}")
    print(df_red.head(3))
except Exception as e:
    print(f"Error al cargar archivo: {e}")

Dataset de red cargado. Forma: (2827562, 87)
                                    flow_id       source_ip  source_port  \
0       172.16.0.1-192.168.10.50-53626-80-6      172.16.0.1        53626   
1  172.217.10.142-192.168.10.15-443-53209-6  172.217.10.142          443   
2   192.168.10.14-104.79.143.90-49502-443-6   104.79.143.90          443   

  destination_ip  destination_port  protocol           timestamp  \
0  192.168.10.50                80         6 2017-07-07 10:09:00   
1  192.168.10.15             53209         6 2017-07-06 16:14:00   
2  192.168.10.14             49502         6 2017-07-06 08:57:00   

   flow_duration  total_fwd_packets  total_backward_packets  ...  active_max  \
0      4941876.0                  4                       0  ...         0.0   
1            3.0                  2                       0  ...         0.0   
2         1322.0                  3                       1  ...         0.0   

   active_min  idle_mean  idle_std  idle_max  idle_min  

In [3]:
# CLASIFICACIÓN DE DATOS DE RED

clasificacion = pd.DataFrame([
    ["flow_id", "Identificador interno", "Confidencial", "Vincular registros", "Seudonimizar"],
    ["source_ip", "Identificador directo", "Confidencial", "Identificación de origen", "Enmascarar octetos"],
    ["destination_ip", "Identificador directo", "Confidencial", "Identificación de destino", "Enmascarar octetos"],
    ["source_port", "Infraestructura", "Interno", "Identificar servicio emisor", "Enmascarar puerto"],
    ["destination_port", "Infraestructura", "Público", "Categorización de servicio", "Conservar"],
    ["protocol", "Técnico/Red", "Público", "Caracterización de tráfico", "Conservar"],
    ["flow_duration", "Métrica de red", "Interno", "Detección de anomalías", "Generalizar por rango"],
    ["total_fwd_packets", "Métrica de red", "Interno", "Medición de volumen", "Conservar"],
    ["min_seg_size_forward", "Métrica de red", "Interno", "Análisis de encabezados", "Conservar"],
    ["active_mean", "Métrica de red", "Interno", "Medición de actividad", "Conservar"],
    ["idle_mean", "Métrica de red", "Interno", "Análisis de pausas", "Generalizar por rango"],
    ["label", "Categoría objetivo", "Público", "Clasificación de ataques", "Conservar"]
], columns=["campo", "tipo", "clasificacion", "finalidad", "control"])

print("Matriz de clasificación ----------")
print(clasificacion)

# GUARDAR CLASIFICACIÓN
ruta_clasificacion = DOCS_DIR / "clasificacion_datos_redes.xlsx"
clasificacion.to_excel(ruta_clasificacion, index=False)

print(f"Clasificación guardada: {ruta_clasificacion}")

Matriz de clasificación ----------
                   campo                   tipo clasificacion  \
0                flow_id  Identificador interno  Confidencial   
1              source_ip  Identificador directo  Confidencial   
2         destination_ip  Identificador directo  Confidencial   
3            source_port        Infraestructura       Interno   
4       destination_port        Infraestructura       Público   
5               protocol            Técnico/Red       Público   
6          flow_duration         Métrica de red       Interno   
7      total_fwd_packets         Métrica de red       Interno   
8   min_seg_size_forward         Métrica de red       Interno   
9            active_mean         Métrica de red       Interno   
10             idle_mean         Métrica de red       Interno   
11                 label     Categoría objetivo       Público   

                      finalidad                control  
0            Vincular registros           Seudonimizar  
1    

In [4]:
# SEUDONIMIZACIÓN

SALT = "proyecto-gobernanza-redes"

def tokenizar(valor):
    """Convierte un valor en un código único de 12 caracteres."""
    texto = f"{SALT}|{valor}".encode('utf-8')
    return hashlib.sha256(texto).hexdigest()[:12]

if 'flow_id' in df_red.columns:
    df_red["flow_token"] = df_red["flow_id"].astype(str).apply(tokenizar)
    print("========Seudonimización aplicada=========")
    print(df_red[["flow_id", "flow_token"]].head())

========Seudonimización aplicada=========
                                    flow_id    flow_token
0       172.16.0.1-192.168.10.50-53626-80-6  5af84e0d4670
1  172.217.10.142-192.168.10.15-443-53209-6  dc250c106b57
2   192.168.10.14-104.79.143.90-49502-443-6  171daaf50a39
3     192.168.10.3-192.168.10.8-53-58834-17  922a79cf8fa7
4     192.168.10.1-192.168.10.3-53-61807-17  b2e7b99a4f4c


In [5]:
# ENMASCARAMIENTO DE IP Y PUERTO

def enmascarar_ip(ip):
    """Oculta los últimos tres octetos de la IP"""
    partes = str(ip).split('.')
    if len(partes) == 4:
        return f"{partes[0]}.***.***.***"
    return "***.***.***.***"

if 'source_ip' in df_red.columns:
    df_red["source_ip_mask"] = df_red["source_ip"].apply(enmascarar_ip)

if 'destination_ip' in df_red.columns:
    df_red["destination_ip_mask"] = df_red["destination_ip"].apply(enmascarar_ip)

if 'source_port' in df_red.columns:
    df_red["source_port_mask"] = df_red["source_port"].astype(str).apply(
        lambda x: "***" + x[-2:] if len(x) >= 2 else "***"
    )

print("IPs y Puertos enmascarados-----------")
print(df_red[["source_ip", "source_ip_mask", "destination_ip", "destination_ip_mask"]].head())

IPs y Puertos enmascarados-----------
        source_ip   source_ip_mask destination_ip destination_ip_mask
0      172.16.0.1  172.***.***.***  192.168.10.50     192.***.***.***
1  172.217.10.142  172.***.***.***  192.168.10.15     192.***.***.***
2   104.79.143.90  104.***.***.***  192.168.10.14     192.***.***.***
3    192.168.10.8  192.***.***.***   192.168.10.3     192.***.***.***
4    192.168.10.3  192.***.***.***   192.168.10.1     192.***.***.***


In [6]:
# GENERALIZACIÓN DE DURACIÓN E IDLE

bins_dur = [-1, 100, 1000, 10000, 100000, 1e9]
labels_dur = ["<=100", "101-1k", "1k-10k", "10k-100k", ">100k"]

if 'flow_duration' in df_red.columns:
    df_red["rango_duracion"] = pd.cut(df_red["flow_duration"], bins=bins_dur, labels=labels_dur)

bins_idle = [-1, 0, 1000, 100000, 1e9]
labels_idle = ["Sin Reposo", "Bajo", "Medio", "Alto"]

if 'idle_mean' in df_red.columns:
    df_red["rango_reposo"] = pd.cut(df_red["idle_mean"], bins=bins_idle, labels=labels_idle)

print("Duración generalizada: ---------------")
print(df_red[["flow_duration", "rango_duracion"]].head())

Duración generalizada: ---------------
   flow_duration rango_duracion
0      4941876.0          >100k
1            3.0          <=100
2         1322.0         1k-10k
3    106084008.0          >100k
4     19917297.0          >100k


In [7]:
# CREAR VISTA ANALÍTICA PROTEGIDA
# PRINCIPIO DE MINIMIZACIÓN

columnas_deseadas = [
    "flow_token",
    "source_ip_mask",
    "destination_ip_mask",
    "source_port_mask",
    "destination_port",
    "protocol",
    "rango_duracion",
    "rango_reposo",
    "total_fwd_packets",
    "min_seg_size_forward",
    "active_mean",
    "label"
]

cols_existentes = [col for col in columnas_deseadas if col in df_red.columns]
vista_analitica = df_red[cols_existentes].copy()

print("VISTA ANALÍTICA PROTEGIDA (SIN IDENTIFICADORES DIRECTOS)")
print(vista_analitica.head())

VISTA ANALÍTICA PROTEGIDA (SIN IDENTIFICADORES DIRECTOS)
     flow_token   source_ip_mask destination_ip_mask source_port_mask  \
0  5af84e0d4670  172.***.***.***     192.***.***.***            ***26   
1  dc250c106b57  172.***.***.***     192.***.***.***            ***43   
2  171daaf50a39  104.***.***.***     192.***.***.***            ***43   
3  922a79cf8fa7  192.***.***.***     192.***.***.***            ***34   
4  b2e7b99a4f4c  192.***.***.***     192.***.***.***            ***07   

   destination_port  protocol rango_duracion rango_reposo  total_fwd_packets  \
0                80         6          >100k   Sin Reposo                  4   
1             53209         6          <=100   Sin Reposo                  2   
2             49502         6         1k-10k   Sin Reposo                  3   
3                53        17          >100k         Alto                  4   
4                53        17          >100k         Alto                  2   

   min_seg_size_forward

In [8]:
# GUARDAR VISTA PROTEGIDA

ruta_publicable = DATA_GOV / "dataset_publicable_redes.csv"
ruta_parquet_pub = DATA_GOV / "dataset_publicable_redes.parquet"

vista_analitica.to_csv(ruta_publicable, index=False, encoding="utf-8")
vista_analitica.to_parquet(ruta_parquet_pub, index=False)

print(f"Vista protegida guardada: {ruta_publicable}")

Vista protegida guardada: ..\Datos\Implementacion_Gobernanza\dataset_publicable_redes.csv


In [9]:
# VALIDACIÓN AUTOMÁTICA

identificadores_directos = {"flow_id", "source_ip", "destination_ip"}
expuestos = identificadores_directos.intersection(vista_analitica.columns)

print("Verificando identificadores directos: --------------")
print(f"Identificadores directos expuestos: {expuestos}")

if len(expuestos) == 0:
    print("Ningún identificador directo expuesto!!!!")
else:
    print("ALERTA: ¡Identificadores directos expuestos!")

Verificando identificadores directos: --------------
Identificadores directos expuestos: set()
Ningún identificador directo expuesto!!!!


In [10]:
# MATRIZ DE RIESGOS

riesgos = pd.DataFrame([
    ["Exposición de direcciones IP internas", 4, 5, "Enmascarar octetos de IP", "Líder de datos", "Dataset protegido"],
    ["Reidentificación por fingerprinting", 3, 4, "Generalizar duración e idle", "Analista", "Análisis de grupos"],
    ["Acceso excesivo al dataset crudo", 3, 5, "RBAC y mínimo privilegio", "Administrador", "Matriz de acceso"],
    ["Retención indefinida de logs", 3, 3, "Regla de retención", "Líder de proyecto", "Bitácora de eliminación"],
    ["Uso del dato para finalidad no definida", 2, 5, "Registrar propósito", "Propietario del dato", "Ficha de finalidad"],
    ["Conclusiones sesgadas por tráfico", 3, 4, "Revisión ética y balanceo", "Equipo analítico", "Reflexión y pruebas"]
], columns=["riesgo", "probabilidad", "impacto", "control", "responsable", "evidencia"])

riesgos["puntaje"] = riesgos["probabilidad"] * riesgos["impacto"]
riesgos["nivel"] = riesgos["puntaje"].apply(
    lambda x: "Crítico" if x >= 16 else "Alto" if x >= 10 else "Moderado" if x >= 5 else "Bajo"
)

print("MATRIZ DE RIESGOS:")
print(riesgos.sort_values("puntaje", ascending=False))

# GUARDAR LA MATRIZ DE RIESGOS
ruta_riesgos = DOCS_DIR / "matriz_riesgos_redes.xlsx"
riesgos.to_excel(ruta_riesgos, index=False)

print(f"Matriz de riesgos guardada---------- {ruta_riesgos}")

MATRIZ DE RIESGOS:
                                    riesgo  probabilidad  impacto  \
0    Exposición de direcciones IP internas             4        5   
2         Acceso excesivo al dataset crudo             3        5   
1      Reidentificación por fingerprinting             3        4   
5        Conclusiones sesgadas por tráfico             3        4   
4  Uso del dato para finalidad no definida             2        5   
3             Retención indefinida de logs             3        3   

                       control           responsable                evidencia  \
0     Enmascarar octetos de IP        Líder de datos        Dataset protegido   
2     RBAC y mínimo privilegio         Administrador         Matriz de acceso   
1  Generalizar duración e idle              Analista       Análisis de grupos   
5    Revisión ética y balanceo      Equipo analítico      Reflexión y pruebas   
4          Registrar propósito  Propietario del dato       Ficha de finalidad   
3          

In [11]:
# POLÍTICA BREVE DE GOBERNANZA

politica = """
# Política de Gobernanza - Network Security Analytics

## 1. Propósito
Definir cómo se utilizan, protegen y comparten los datos de tráfico de red del proyecto.

## 2. Clasificación
- Público: puertos estándar, etiquetas de tráfico (label)
- Interno: métricas de volumen y tiempos (duration, active, idle)
- Confidencial: direcciones IP (origen y destino), flow_id
- Restringido: claves de salado (SALT) y logs crudos

## 3. Acceso por Rol
- Administrador: acceso total a datos crudos
- Analista / Científico de datos: solo vista protegida (sin identificadores)
- Auditor: solo lectura de evidencias

## 4. Minimización
Solo se incluyen los campos estrictamente necesarios para el entrenamiento de modelos IDS.

## 5. Retención
Los datos se conservan durante el ciclo del proyecto y se eliminan al finalizar.

## 6. Ética
No se utilizan direcciones IP para desanonimizar equipos ni usuarios en la red.
"""

# GUARDAR POLÍTICA
ruta_politica = DOCS_DIR / "politica_gobernanza_redes.md"
with open(ruta_politica, "w", encoding="utf-8") as f:
    f.write(politica)

print(f"Política guardada: {ruta_politica}")

Política guardada: ..\Datos\CSV_Generados\politica_gobernanza_redes.md


In [12]:

# VERIFICAR ARCHIVOS GENERADOS
print("====ARCHIVOS GENERADOS====")
print("=" * 40)
for archivo in DATA_GOV.glob("*"):
  print(f"{archivo.name:<40} {archivo.stat().st_size / 1024:.2f} KB")

print("\n====DOCUMENTOS====")
for documento in DOCS_DIR.glob("*"):
  print(f"{documento.name:<40} {documento.stat().st_size / 1024:.2f} KB")

====ARCHIVOS GENERADOS====
.gitkeep                                 0.00 KB
dataset_publicable_redes.csv             259463.79 KB
dataset_publicable_redes.parquet         57045.63 KB

====DOCUMENTOS====
.gitkeep                                 0.00 KB
clasificacion_datos_redes.xlsx           5.38 KB
matriz_riesgos_redes.xlsx                5.37 KB
politica_gobernanza_redes.md             0.92 KB
